# Model Parameters Optimization

## Getting Data + Target encoding

In [ ]:
import pandas as pd

data = pd.read_csv('Data/trainVal.csv', index_col=False)
X_trainval, y_trainval = data.drop('Air_Quality', axis=1), data['Air_Quality']

# map = {'Hazardous': 0, 'Poor': 1, 'Moderate': 2, 'Good': 3} # Ordinal Encoding (as data is Ordinal) / other approach
# y_trainval = y_trainval.map(map)

y_trainval = pd.get_dummies(y_trainval) # one hot encoding - no bias, we don't assume anything about data

y_trainval

In [ ]:

from utils.training import get_dataSet

dataset = get_dataSet(X_trainval, y_trainval)

type(dataset)

# Optuna Study

In [ ]:
import os
import shutil
from torch.utils.tensorboard import SummaryWriter

writer_path = os.path.join('runs', 'optuna')
if os.path.exists(writer_path):
    print(f"Cleaning existing logs at {writer_path}...")
    shutil.rmtree(writer_path)

os.makedirs(writer_path, exist_ok=True)
writer = SummaryWriter(writer_path)

## Objective
**Prompt**:
```
For 4000 instance trianval dataset, suggest best ranges for this parameters.
The network is optimized with 5 fold cross validation by mean validation loss score.

Params to optimize ...
```

| Parameter | Suggested Range | Reasoning |
| :--- | :--- | :--- |
| **`lr`** | `1e-4` to `1e-2` | `5e-1` (0.5) is extremely high and will likely cause the loss to explode. |
| **`batch_size`** | `16` to `128` | With 4,000 samples, a batch of 1 is too noisy; 256 is nearly 10% of your training fold. |
| **`beta1`** | `0.85` to `0.95` | This controls momentum. The default is 0.9. |
| **`beta2`** | `0.99` to `0.9999` | This controls the moving average of squared gradients. |

The goal is to find best parameters for **100** Epochs

In [ ]:
# Paste this to console to see Tenosrboard
# tensorboard --logdir Lab6_7-project/runs/optuna

In [ ]:
import numpy as np
import optuna
from functools import partial

from torch import nn
from torch import optim
from torch.utils.data import TensorDataset
from sklearn.model_selection import KFold

from utils.training import *
from utils.MLPClassifier import MLPClassifier

N_EPOCHS = 100
optuna.logging.set_verbosity(optuna.logging.WARNING) # No Reason to se results as we have tensorboard
def objective_nn(trial:optuna.Trial, dataset:TensorDataset, input_dim:int, output_dim:int, writer:SummaryWriter):
    global N_EPOCHS

    # Network Params
    # Using step=16 or 32 helps keep dimensions hardware-friendly
    hidden_dim = trial.suggest_int('hidden_dim', 32, 512, step=32)
    n_hidden = trial.suggest_int('n_hidden', 1, 5)

    # Optimizer Params
    lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical('batch_size', [16, 32, 64, 128])

    # If using Adam, betas are usually stable.
    # Only tune them if you have a specific reason to deviate from defaults.
    beta1 = trial.suggest_float('beta1', 0.8, 0.99)
    beta2 = trial.suggest_float('beta2', 0.99, 0.9999)

    # Scheduler Params
    factor = trial.suggest_float('factor', 0.1, 0.5)
    patience = trial.suggest_int('patience', 3, 7)


    fold_loss = []
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    for fold, (train_idx, val_idx) in enumerate(kf.split(X_trainval)):

        # Loaders
        train_loader, val_loader = get_train_loaders(dataset, train_idx=train_idx, val_idx=val_idx, batch_size=batch_size)

        # Model, Optimizer, Criterion
        model = MLPClassifier(input_dim=input_dim, output_dim=output_dim, hidden_dim=hidden_dim, n_hidden=n_hidden)
        criterion = nn.CrossEntropyLoss() # nn.MSELoss (better if we choose ordinal encoding)
        optimizer = optim.Adam(lr=lr, params=model.parameters(), betas=(beta1, beta2))
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=factor, patience=patience)

        # Fold Training
        loss = train_one_fold(fold, model, train_loader, val_loader, optimizer, scheduler, criterion, n_epochs=N_EPOCHS, write_model_dir=None, writer=None)
        fold_loss.append(loss)

    avg_loss = np.mean(fold_loss)
    # Writing Params
    writer.add_hparams(
        hparam_dict={
        'trial': trial.number,
        'hidden_dim': hidden_dim,
        'n_hidden': n_hidden,
        'lr': lr,
        'batch_size': batch_size,
        'beta1': beta1,
        'beta2': beta2,
        'factor': factor,
        'patience': patience
    },
    metric_dict={'hparam/mean_loss': avg_loss})
    writer.flush()


    return avg_loss

objective = partial(objective_nn, dataset=dataset, input_dim=X_trainval.shape[1], output_dim=y_trainval.shape[1], writer=writer)
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=25, show_progress_bar=True)

## Results

In [ ]:
import pandas as pd
trials = study.trials_dataframe().sort_values(by=['value'], ascending=True)

trials.head(5)

In [ ]:
from datetime import datetime

now = datetime.now()
formatted_date = now.strftime("%d_%m_%Y(%H:%M)")

path = os.path.join('optuna_results', f'trial_{formatted_date}.csv')
os.makedirs('optuna_results', exist_ok=True)
trials.to_csv(path, index=False)